In [17]:
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import models, layers

In [18]:
from tensorflow.keras.datasets import mnist
import numpy
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical


In [19]:
(train_images,train_labels),(test_images,test_labels)=mnist.load_data()

In [20]:
train_images[0].min()

0

In [21]:
train_images=train_images.astype('float32')/255
test_images=test_images.astype('float32')/255

In [22]:
train_images=train_images.reshape(train_images.shape[0],28,28,1) #the model cannot read grayscale, therefore reshape

In [23]:
test_images=test_images.reshape(test_images.shape[0],28,28,1)

In [24]:
train_images[0].shape

(28, 28, 1)

Data Augumentation
If we don't have enough train data (images) to work on then we generate more data using data augumentation. It flips and rotates images randomly

In [25]:
data_augmentation=tf.keras.Sequential([layers.RandomRotation(0.1),
                                      layers.RandomFlip(),
                                      layers.RandomZoom(0.1),
                                      layers.RandomTranslation(0.1,0.1)])

In [30]:
def create_model():
    inputs = keras.Input(shape=(28,28,1))
    x = data_augmentation(inputs)

    x = keras.layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPool2D(2)(x)
    x = keras.layers.SpatialDropout2D(0.2)(x)


    x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPool2D(2)(x)
    x = keras.layers.SpatialDropout2D(0.2)(x)

    residual = x
    x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.add([x,residual])

    
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    outputs = keras.layers.Dense(10, activation="softmax")(x)

    model =keras.Model(inputs, outputs)
    return model

In [31]:
from tensorflow.keras.callbacks import EarlyStopping

In [32]:
callbacks = [EarlyStopping(patience=10, restore_best_weights= True)]

In [33]:
model=create_model()

In [34]:
from tensorflow.keras.optimizers import Adam

In [35]:
model.compile(optimizer=Adam(learning_rate=0.001),loss="sparse_categorical_crossentropy",metrics=['accuracy'])


In [36]:
history = model.fit(train_images, train_labels, callbacks=callbacks, epochs=2, batch_size=8, )

Epoch 1/2
7500/7500 ━━━━━━━━━━━━━━━━━━━━ 190s 24ms/step - accuracy: 0.6536 - loss: 0.9854
Epoch 2/2
   4/7500 ━━━━━━━━━━━━━━━━━━━━ 2:56 24ms/step - accuracy: 0.8802 - loss: 0.4683

C:\Users\Shreya Ramachandran\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\callbacks\early_stopping.py:155: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


7500/7500 ━━━━━━━━━━━━━━━━━━━━ 193s 26ms/step - accuracy: 0.8757 - loss: 0.3786


In [37]:
from sklearn.metrics import classification_report

In [38]:
import pprint

In [39]:
y_pred= model.predict(test_images).argmax(axis=1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step


In [40]:
pprint.pprint(classification_report(test_labels, y_pred))

('              precision    recall  f1-score   support\n'
 '\n'
 '           0       1.00      0.98      0.99       980\n'
 '           1       1.00      0.97      0.98      1135\n'
 '           2       0.94      0.88      0.91      1032\n'
 '           3       0.98      0.98      0.98      1010\n'
 '           4       0.98      0.97      0.98       982\n'
 '           5       0.87      0.96      0.91       892\n'
 '           6       0.94      0.88      0.91       958\n'
 '           7       0.93      0.97      0.95      1028\n'
 '           8       0.96      0.98      0.97       974\n'
 '           9       0.91      0.92      0.91      1009\n'
 '\n'
 '    accuracy                           0.95     10000\n'
 '   macro avg       0.95      0.95      0.95     10000\n'
 'weighted avg       0.95      0.95      0.95     10000\n')
